# Model Evaluation
Evaluate the saved model on a held-out dataset using metrics appropriate to your target type. Keep evaluation data separate from training data.

In [ ]:
#import libraries 
import seaborn as sns

from sklearn.metrics import (
 accuracy_score,
 classification_report,
 confusion_matrix,
 mean_absolute_error,
 mean_squared_error,
 r2_score
)
from sklearn.model_selection import train_test_split



In [11]:
from pathlib import Path
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

MODEL_PATH = PROJECT_ROOT / 'models' / 'irrigation_model.joblib'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

TARGET_COLUMN = 'Irrigation_Need'
print("Paths initialized successfully!")
print("Project Root:", PROJECT_ROOT)
print("Model Path:", MODEL_PATH)
print("Processed Data Directory:", PROCESSED_DATA_DIR)

Paths initialized successfully!
Project Root: g:\AIOT\smart-irrigation
Model Path: g:\AIOT\smart-irrigation\models\irrigation_model.joblib
Processed Data Directory: g:\AIOT\smart-irrigation\data\processed


In [12]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

TARGET_COLUMN = 'Irrigation_Need'
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_PATH = PROJECT_ROOT / "models" / "irrigation_model.joblib"

# Look for a dedicated test file first, fallback to processed files
test_file = PROCESSED_DATA_DIR / 'test.csv'
if not test_file.is_file():
  processed_files = sorted(PROCESSED_DATA_DIR.glob('*.csv'))
  if not processed_files:
    raise FileNotFoundError(
        'Train a model and create processed data before evaluation.'
    )
  test_file = processed_files[0]

if not MODEL_PATH.is_file():
  raise FileNotFoundError(f'Model not found at {MODEL_PATH}.')

data = pd.read_csv(test_file)

if TARGET_COLUMN not in data.columns:
  raise ValueError(
      f"Target column '{TARGET_COLUMN}' not found in dataset columns:"
      f' {list(data.columns)}'
  )

X_test = data.drop(columns=[TARGET_COLUMN]).select_dtypes(include='number')
y_test = data[TARGET_COLUMN]

# Load model
model = joblib.load(MODEL_PATH)

# Validate feature consistency if the model exposes feature names
if hasattr(model, 'feature_names_in_'):
  missing_cols = set(model.feature_names_in_) - set(X_test.columns)
  if missing_cols:
    raise ValueError(f'Missing expected features in test data: {missing_cols}')
  X_test = X_test[model.feature_names_in_]

predictions = model.predict(X_test)

# Evaluate based on target data type
if y_test.dtype.kind in 'if':
  metrics = {
      'mae': mean_absolute_error(y_test, predictions),
      'rmse': mean_squared_error(y_test, predictions) ** 0.5,
      'r2': r2_score(y_test, predictions),
  }
  print('Regression Metrics:')
  for metric, value in metrics.items():
    print(f'  {metric.upper():<5}: {value:.4f}')
else:
  print(f'Accuracy: {accuracy_score(y_test, predictions):.3f}\n')
  print('Classification Report:')
  print(classification_report(y_test, predictions))

Accuracy: 0.946

Classification Report:
              precision    recall  f1-score   support

        High       0.97      0.88      0.92       336
         Low       0.95      0.97      0.96      5864
      Medium       0.94      0.92      0.93      3800

    accuracy                           0.95     10000
   macro avg       0.95      0.92      0.94     10000
weighted avg       0.95      0.95      0.95     10000



In [4]:
from pathlib import Path

MODEL_PATH = Path("models/irrigation_model.pkl")  # Update to your actual path
print("Model path:", MODEL_PATH)
print("Does it exist?", MODEL_PATH.is_file())

Model path: models\irrigation_model.pkl
Does it exist? False


In [ ]:
model_files = list(Path(".").rglob("*.pkl")) + list(Path(".").rglob("*.joblib"))

print("Found model files:")
for path in model_files:
  print(f" - {path.resolve()}")

Found model files:
